# FinanceToolkit Feature Family Predictive Power EDA

This notebook treats FinanceToolkit as an FMP-oriented computation library and asks a practical question: if we organize features by FinanceToolkit-style modules, which families show predictive power against forward equity returns?

The notebook does not call raw FMP APIs directly. It uses OpenBB/FMP only for universe screening, then reads historical prices and fundamentals from Quant Warehouse storage.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import importlib
import inspect
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

_current = Path.cwd().resolve()
_repo_candidates = [_current, *_current.parents]
REPO_ROOT = next((path for path in _repo_candidates if (path / "quant_warehouse").exists() and (path / "pyproject.toml").exists()), _current)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse import Warehouse
from quant_warehouse.ingest.credentials import configure_openbb_credentials

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROVIDER = "fmp"
ANALYSIS_LABEL = "OpenBB/FMP screened US >= $100B universe"
SCREEN_MARKET_CAP_MIN = 100_000_000_000
SCREEN_COUNTRY = "US"
SCREEN_EXCHANGES = ("NASDAQ", "NYSE", "AMEX")
SCREEN_LIMIT = 5_000
START_DATE = "2018-01-01"
END_DATE = None
FILING_LAG_DAYS = 45
HORIZONS = (20, 60, 120)
MIN_OBS = 120

wh = Warehouse()
run_timings: dict[str, float] = {}


## FinanceToolkit Module Surface

The first table inspects the installed FinanceToolkit package and records the public methods exposed by the major computation controllers. These method groups drive the feature-family taxonomy used below.


In [2]:
import financetoolkit

module_status = pd.DataFrame([{
    "module": "financetoolkit",
    "installed": True,
    "version": getattr(financetoolkit, "__version__", "unknown"),
    "module_file": getattr(financetoolkit, "__file__", None),
}])
display(module_status)

controller_classes = [
    ("ratios", "financetoolkit.ratios.ratios_controller", "Ratios"),
    ("models", "financetoolkit.models.models_controller", "Models"),
    ("performance", "financetoolkit.performance.performance_controller", "Performance"),
    ("risk", "financetoolkit.risk.risk_controller", "Risk"),
    ("technicals", "financetoolkit.technicals.technicals_controller", "Technicals"),
]
controller_methods = []
for controller_name, module_name, class_name in controller_classes:
    try:
        module = importlib.import_module(module_name)
        controller_class = getattr(module, class_name)
        methods = [name for name in dir(controller_class) if not name.startswith("_") and callable(getattr(controller_class, name, None))]
        controller_methods.append({
            "controller": controller_name,
            "method_count": len(methods),
            "sample_methods": tuple(methods[:15]),
        })
    except Exception as exc:
        controller_methods.append({
            "controller": controller_name,
            "method_count": 0,
            "sample_methods": (f"{type(exc).__name__}: {exc}",),
        })

controller_methods_df = pd.DataFrame(controller_methods)
display(controller_methods_df)


,module,installed,version,module_file
0,financetoolkit,True,unknown,/home/jlee153232/miniconda3/lib/python3.13/sit...


,controller,method_count,sample_methods
0,ratios,77,"(collect_all_ratios, collect_custom_ratios, co..."
1,models,9,"(get_altman_z_score, get_dupont_analysis, get_..."
2,performance,16,"(collect_all_metrics, get_alpha, get_beta, get..."
3,risk,10,"(collect_all_metrics, get_conditional_value_at..."
4,technicals,36,"(collect_all_indicators, collect_breadth_indic..."


## Universe

Use the same provider-screened large-cap universe as the statement notebooks, then require stored Quant Warehouse history for prices and the fundamental sections used by the feature families.


In [3]:
FALLBACK_SYMBOLS = (
    "AAPL", "ABBV", "ABT", "ADI", "AMAT", "AMD", "AMGN", "AMZN", "ANET", "APH", "APP", "AVGO", "AXP", "BA", "BAC", "BKNG", "BLK", "BMY", "BRK-A", "BRK-B", "BX", "C", "CAT", "CDNS", "COF", "COP", "COST", "CRM", "CRWD", "CSCO", "CVS", "CVX", "DE", "DELL", "DHR", "DIS", "DUK", "EQIX", "FTNT", "GE", "GEV", "GILD", "GLW", "GOOG", "GOOGL", "GS", "HD", "HONIV", "HWM", "IBKR", "IBM", "INTC", "ISRG", "JNJ", "JPM", "KLAC", "KO", "LLY", "LMT", "LOW", "LRCX", "MA", "MCD", "MDT", "META", "MO", "MRK", "MRVL", "MS", "MSFT", "MU", "NEE", "NEM", "NFLX", "NOW", "NVDA", "ORCL", "PANW", "PEP", "PFE", "PG", "PGR", "PH", "PLD", "PLTR", "PM", "PWR", "QCOM", "RTX", "SBUX", "SCCO", "SCHW", "SNDK", "SO", "SOJE", "SOMN", "SPGI", "SYK", "T", "TBB", "TJX", "TMO", "TMUS", "TSLA", "TXN", "UBER", "UNH", "UNP", "V", "VRT", "VRTX", "VZ", "WDC", "WELL", "WFC", "WMT", "XOM"
)
REQUIRED_HISTORY_SECTIONS = ("prices", "income", "balance", "cash", "ratios", "metrics", "income_growth", "balance_growth", "cash_growth")


def _read_history_section(symbol: str, section: str) -> pd.DataFrame:
    if section == "prices":
        return wh.read_prices(symbol, provider=PROVIDER)
    return wh.read_fundamentals(symbol, section=section, provider=PROVIDER)


def has_required_warehouse_history(symbol: str) -> tuple[bool, str]:
    for section in REQUIRED_HISTORY_SECTIONS:
        try:
            frame = _read_history_section(symbol, section)
        except Exception as exc:
            return False, f"{section}: {type(exc).__name__}"
        if frame is None or frame.empty:
            return False, f"{section}: empty"
    return True, "ok"


def _normalize_screener_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c).strip() for c in out.columns]
    if "symbol" not in out.columns:
        for candidate in ("ticker", "Symbol", "Ticker"):
            if candidate in out.columns:
                out = out.rename(columns={candidate: "symbol"})
                break
    if "symbol" not in out.columns:
        return pd.DataFrame(columns=["symbol"])
    out["symbol"] = out["symbol"].astype(str).str.strip().str.upper()
    return out.dropna(subset=["symbol"]).drop_duplicates("symbol")


def fetch_openbb_fmp_screener(exchange: str) -> pd.DataFrame:
    from openbb import obb

    configure_openbb_credentials()
    data = obb.equity.search(
        provider="fmp",
        mktcap_min=SCREEN_MARKET_CAP_MIN,
        country=SCREEN_COUNTRY,
        exchange=exchange,
        is_etf=False,
        is_fund=False,
        is_active=True,
        all_share_classes=False,
        limit=SCREEN_LIMIT,
    )
    frame = data.to_df() if hasattr(data, "to_df") else pd.DataFrame(data)
    return _normalize_screener_frame(frame)


fetch_rows = []
frames = []
try:
    for exchange in SCREEN_EXCHANGES:
        try:
            frame = fetch_openbb_fmp_screener(exchange)
            fetch_rows.append({"exchange": exchange, "source": "openbb:fmp", "rows": len(frame), "status": "ok" if len(frame) else "empty"})
            frames.append(frame)
        except Exception as exc:
            fetch_rows.append({"exchange": exchange, "source": "openbb:fmp", "rows": 0, "status": type(exc).__name__})
    raw_universe = pd.concat(frames, ignore_index=True).drop_duplicates("symbol") if frames else pd.DataFrame(columns=["symbol"])
except Exception:
    raw_universe = pd.DataFrame(columns=["symbol"])

if raw_universe.empty:
    raw_universe = pd.DataFrame({"symbol": FALLBACK_SYMBOLS})
    fetch_rows.append({"exchange": "fallback", "source": "stored_previous_screen", "rows": len(raw_universe), "status": "used"})

eligibility_rows = []
for symbol in raw_universe["symbol"].astype(str).str.upper().drop_duplicates():
    ok, reason = has_required_warehouse_history(symbol)
    eligibility_rows.append({"symbol": symbol, "eligible": ok, "reason": reason})

eligibility = pd.DataFrame(eligibility_rows)
ANALYSIS_SYMBOLS = tuple(eligibility.loc[eligibility["eligible"], "symbol"].sort_values())
if not ANALYSIS_SYMBOLS:
    raise RuntimeError("No eligible symbols found for FinanceToolkit family EDA.")

display(pd.DataFrame(fetch_rows))
display(eligibility["reason"].value_counts().rename_axis("reason").reset_index(name="symbols"))
display(Markdown(f"> Selected {len(ANALYSIS_SYMBOLS):,} symbols from `{ANALYSIS_LABEL}` after warehouse-history checks."))


,exchange,source,rows,status
0,NASDAQ,openbb:fmp,0,OpenBBError
1,NYSE,openbb:fmp,0,OpenBBError
2,AMEX,openbb:fmp,0,OpenBBError
3,fallback,stored_previous_screen,117,used


,reason,symbols
0,ok,117


> Selected 117 symbols from `OpenBB/FMP screened US >= $100B universe` after warehouse-history checks.

## Feature Family Construction

The families are based on FinanceToolkit controller groups:

- `ft_ratios_*`: profitability, liquidity, solvency, efficiency, and valuation families from FMP ratio/metric sections.
- `ft_growth_*`: income, balance, and cash growth families from FMP growth sections.
- `ft_technicals_*`, `ft_performance_price`, and `ft_risk_price`: daily price-derived families aligned to FinanceToolkit's technicals, performance, and risk controllers.

Quarterly fundamentals are shifted by a 45-day filing lag before being broadcast to daily rows.


In [4]:
@dataclass(frozen=True)
class FeatureSpec:
    feature: str
    family: str
    controller: str
    source_section: str
    source_column: str
    expected_direction: str


def _slice(frame: pd.DataFrame, start: str | None, end: str | None) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    if start is not None:
        out = out.loc[out.index >= pd.Timestamp(start)]
    if end is not None:
        out = out.loc[out.index <= pd.Timestamp(end)]
    return out


def _numeric_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.apply(pd.to_numeric, errors="coerce")
    return out.replace([np.inf, -np.inf], np.nan)


def _align_fundamental(frame: pd.DataFrame, daily_index: pd.DatetimeIndex, *, lag_days: int = FILING_LAG_DAYS) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame(index=daily_index)
    sparse = _numeric_frame(frame)
    sparse.index = pd.DatetimeIndex(pd.to_datetime(sparse.index, errors="coerce")).normalize() + pd.Timedelta(days=int(lag_days))
    sparse = sparse.loc[sparse.index.notna()].sort_index()
    sparse = sparse.loc[~sparse.index.duplicated(keep="last")]
    return sparse.reindex(daily_index, method="ffill")


def _family_for_ratio_column(column: str) -> str | None:
    c = column.lower()
    if any(token in c for token in ["margin", "return_on", "income_quality", "tax_burden", "interest_burden", "sga_to_revenue"]):
        return "ft_ratios_profitability"
    if any(token in c for token in ["current_ratio", "quick_ratio", "cash_ratio", "operating_cash_flow_ratio", "working_capital"]):
        return "ft_ratios_liquidity"
    if any(token in c for token in ["debt", "coverage", "equity_multiplier", "liabilities", "solvency"]):
        return "ft_ratios_solvency"
    if any(token in c for token in ["turnover", "days_", "cycle", "cash_conversion", "asset_turnover"]):
        return "ft_ratios_efficiency"
    if any(token in c for token in ["price_to", "ev_to", "market_cap", "enterprise_value", "yield", "book_value_per_share", "earnings_per_share", "revenue_per_share", "capex_per_share"]):
        return "ft_ratios_valuation"
    return None


def _family_for_metric_column(column: str) -> str | None:
    c = column.lower()
    if any(token in c for token in ["ev_to", "market_cap", "enterprise_value", "graham", "invested_capital"]):
        return "ft_ratios_valuation"
    if any(token in c for token in ["return_on", "income_quality", "roic"]):
        return "ft_ratios_profitability"
    if any(token in c for token in ["debt", "coverage", "working_capital"]):
        return "ft_ratios_solvency"
    return None


def _expected_direction(family: str, column: str) -> str:
    c = column.lower()
    if family in {"ft_ratios_valuation", "ft_ratios_solvency", "ft_risk_price"}:
        if any(token in c for token in ["yield", "cash", "working_capital", "interest_coverage", "current_ratio", "quick_ratio"]):
            return "higher_is_better"
        return "lower_is_better"
    if family in {"ft_technicals_overlap"}:
        return "lower_is_better" if any(token in c for token in ["distance_above", "zscore"]) else "higher_is_better"
    return "higher_is_better"


def load_symbol_inputs(symbol: str) -> dict[str, pd.DataFrame]:
    return {
        "prices": _slice(wh.read_prices(symbol, provider=PROVIDER), START_DATE, END_DATE),
        "income": _slice(wh.read_fundamentals(symbol, section="income", provider=PROVIDER), None, END_DATE),
        "balance": _slice(wh.read_fundamentals(symbol, section="balance", provider=PROVIDER), None, END_DATE),
        "cash": _slice(wh.read_fundamentals(symbol, section="cash", provider=PROVIDER), None, END_DATE),
        "ratios": _slice(wh.read_fundamentals(symbol, section="ratios", provider=PROVIDER), None, END_DATE),
        "metrics": _slice(wh.read_fundamentals(symbol, section="metrics", provider=PROVIDER), None, END_DATE),
        "income_growth": _slice(wh.read_fundamentals(symbol, section="income_growth", provider=PROVIDER), None, END_DATE),
        "balance_growth": _slice(wh.read_fundamentals(symbol, section="balance_growth", provider=PROVIDER), None, END_DATE),
        "cash_growth": _slice(wh.read_fundamentals(symbol, section="cash_growth", provider=PROVIDER), None, END_DATE),
    }


In [5]:
def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.divide(denominator.replace(0.0, np.nan))


def _rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def _max_drawdown(close: pd.Series, window: int) -> pd.Series:
    peak = close.rolling(window, min_periods=max(5, window // 4)).max()
    return close / peak - 1.0


def _price_feature_frame(prices: pd.DataFrame) -> tuple[pd.DataFrame, list[FeatureSpec]]:
    specs: list[FeatureSpec] = []
    close = pd.to_numeric(prices["close"], errors="coerce")
    volume = pd.to_numeric(prices.get("volume", pd.Series(index=prices.index, dtype="float64")), errors="coerce")
    high = pd.to_numeric(prices.get("high", close), errors="coerce")
    low = pd.to_numeric(prices.get("low", close), errors="coerce")
    returns = close.pct_change()
    out = pd.DataFrame(index=prices.index)

    def add(name: str, values: pd.Series, family: str, source_column: str, expected_direction: str | None = None) -> None:
        feature = f"{family}__{name}"
        out[feature] = values
        specs.append(FeatureSpec(feature, family, family.split("_")[1], "prices", source_column, expected_direction or _expected_direction(family, name)))

    for window in (20, 60, 120):
        add(f"return_{window}d", close.pct_change(window), "ft_performance_price", f"close.pct_change({window})", "higher_is_better")
        vol = returns.rolling(window).std() * np.sqrt(252)
        add(f"volatility_{window}d", vol, "ft_risk_price", f"returns.rolling({window}).std", "lower_is_better")
        add(f"sharpe_like_{window}d", returns.rolling(window).mean() / returns.rolling(window).std().replace(0.0, np.nan), "ft_performance_price", f"returns.rolling({window}).mean/std", "higher_is_better")
        add(f"max_drawdown_{window}d", _max_drawdown(close, window), "ft_risk_price", f"rolling({window}).drawdown", "higher_is_better")
        ma = close.rolling(window).mean()
        add(f"distance_above_sma_{window}d", _safe_divide(close - ma, ma), "ft_technicals_overlap", f"close/sma_{window}", "lower_is_better")
        add(f"volume_zscore_{window}d", (volume - volume.rolling(window).mean()) / volume.rolling(window).std().replace(0.0, np.nan), "ft_technicals_volume", f"volume.zscore({window})", "higher_is_better")

    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    add("rsi_14", _rsi(close, 14), "ft_technicals_momentum", "rsi_14", "lower_is_better")
    add("macd_to_price", _safe_divide(macd, close), "ft_technicals_momentum", "macd/close", "higher_is_better")
    add("bollinger_zscore_20d", (close - close.rolling(20).mean()) / close.rolling(20).std().replace(0.0, np.nan), "ft_technicals_volatility", "bollinger_zscore_20", "lower_is_better")
    true_range = pd.concat([(high - low).abs(), (high - close.shift()).abs(), (low - close.shift()).abs()], axis=1).max(axis=1)
    add("atr_to_price_14d", _safe_divide(true_range.rolling(14).mean(), close), "ft_technicals_volatility", "atr_14/close", "lower_is_better")
    add("downside_volatility_60d", returns.where(returns < 0).rolling(60).std() * np.sqrt(252), "ft_risk_price", "downside_vol_60", "lower_is_better")
    add("var_60d_05", returns.rolling(60).quantile(0.05), "ft_risk_price", "var_60_05", "higher_is_better")
    return out, specs


def build_symbol_panel(symbol: str) -> tuple[pd.DataFrame, list[FeatureSpec], dict[str, object]]:
    start = perf_counter()
    inputs = load_symbol_inputs(symbol)
    prices = inputs["prices"]
    if prices.empty or "close" not in prices.columns:
        return pd.DataFrame(), [], {"symbol": symbol, "status": "missing_prices"}

    daily_index = pd.DatetimeIndex(prices.index)
    panel = pd.DataFrame({"date": daily_index, "symbol": symbol, "close": pd.to_numeric(prices["close"], errors="coerce")}, index=daily_index)
    feature_frames: dict[str, pd.Series] = {}
    specs: list[FeatureSpec] = []

    price_features, price_specs = _price_feature_frame(prices)
    feature_frames.update({col: price_features[col] for col in price_features.columns})
    specs.extend(price_specs)

    aligned = {section: _align_fundamental(inputs[section], daily_index) for section in ["ratios", "metrics", "income_growth", "balance_growth", "cash_growth"]}
    for section in ["ratios", "metrics"]:
        frame = aligned[section]
        for column in frame.columns:
            family = _family_for_ratio_column(column) if section == "ratios" else _family_for_metric_column(column)
            if family is None:
                continue
            feature = f"{family}__{column}"
            feature_frames[feature] = frame[column]
            specs.append(FeatureSpec(feature, family, "ratios", section, column, _expected_direction(family, column)))

    for section, family in [("income_growth", "ft_growth_income"), ("balance_growth", "ft_growth_balance"), ("cash_growth", "ft_growth_cash")]:
        frame = aligned[section]
        for column in frame.columns:
            if not str(column).startswith("growth_"):
                continue
            feature = f"{family}__{column}"
            feature_frames[feature] = frame[column]
            specs.append(FeatureSpec(feature, family, "performance", section, column, "higher_is_better"))

    if not feature_frames:
        return pd.DataFrame(), [], {"symbol": symbol, "status": "no_features"}
    feature_df = pd.DataFrame(feature_frames, index=daily_index)
    panel = pd.concat([panel, feature_df], axis=1)
    for horizon in HORIZONS:
        panel[f"forward_return_{horizon}d"] = panel["close"].shift(-horizon) / panel["close"] - 1.0
    return panel.reset_index(drop=True), specs, {
        "symbol": symbol,
        "status": "ok",
        "price_rows": len(prices),
        "feature_count": len(feature_frames),
        "source_align_seconds": perf_counter() - start,
    }


## Build Daily Feature Panel


In [6]:
panel_build_start = perf_counter()
frames: list[pd.DataFrame] = []
all_specs: list[FeatureSpec] = []
diagnostics: list[dict[str, object]] = []
for symbol in ANALYSIS_SYMBOLS:
    frame, specs, diag = build_symbol_panel(symbol)
    diagnostics.append(diag)
    if not frame.empty:
        frames.append(frame)
        all_specs.extend(specs)

panel = pd.concat(frames, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
feature_metadata = pd.DataFrame([spec.__dict__ for spec in all_specs]).drop_duplicates().sort_values(["family", "feature"]).reset_index(drop=True)
diagnostics_df = pd.DataFrame(diagnostics)
feature_cols = feature_metadata["feature"].tolist()
run_timings["panel_build_seconds"] = perf_counter() - panel_build_start
run_timings["source_align_seconds"] = float(diagnostics_df.get("source_align_seconds", pd.Series(dtype="float64")).sum())

summary = {
    "symbols": int(panel["symbol"].nunique()),
    "rows": int(len(panel)),
    "features": int(len(feature_cols)),
    "start": str(panel["date"].min().date()),
    "end": str(panel["date"].max().date()),
    "panel_build_seconds": round(run_timings["panel_build_seconds"], 2),
    "source_align_seconds": round(run_timings["source_align_seconds"], 2),
}
print(summary)
display(diagnostics_df)
display(feature_metadata.groupby(["controller", "family"]).size().rename("feature_count").reset_index().sort_values(["controller", "family"]))
display(feature_metadata.head(60))


{'symbols': 117, 'rows': 238717, 'features': 244, 'start': '2018-01-02', 'end': '2026-06-24', 'panel_build_seconds': 6.22, 'source_align_seconds': 5.92}


,symbol,status,price_rows,feature_count,source_align_seconds
0,AAPL,ok,2130,244,0.0775
1,ABBV,ok,2130,244,0.0476
2,ABT,ok,2130,244,0.0498
3,ADI,ok,2130,244,0.0427
4,AMAT,ok,2130,244,0.0430
...,...,...,...,...,...
112,WDC,ok,2130,244,0.0527
113,WELL,ok,2130,244,0.0546
114,WFC,ok,2130,244,0.0646
115,WMT,ok,2130,244,0.0509


,controller,family,feature_count
0,performance,ft_growth_balance,56
1,performance,ft_growth_cash,50
2,performance,ft_growth_income,38
3,performance,ft_performance_price,6
4,ratios,ft_ratios_efficiency,5
5,ratios,ft_ratios_liquidity,7
6,ratios,ft_ratios_profitability,14
7,ratios,ft_ratios_solvency,19
8,ratios,ft_ratios_valuation,31
9,risk,ft_risk_price,8


,feature,family,controller,source_section,source_column,expected_direction
0,ft_growth_balance__growth_account_payables,ft_growth_balance,performance,balance_growth,growth_account_payables,higher_is_better
1,ft_growth_balance__growth_accounts_receivables,ft_growth_balance,performance,balance_growth,growth_accounts_receivables,higher_is_better
2,ft_growth_balance__growth_accrued_expenses,ft_growth_balance,performance,balance_growth,growth_accrued_expenses,higher_is_better
3,ft_growth_balance__growth_accumulated_other_co...,ft_growth_balance,performance,balance_growth,growth_accumulated_other_comprehensive_income,higher_is_better
4,ft_growth_balance__growth_accumulated_other_co...,ft_growth_balance,performance,balance_growth,growth_accumulated_other_comprehensive_income_...,higher_is_better
5,ft_growth_balance__growth_additional_paid_in_c...,ft_growth_balance,performance,balance_growth,growth_additional_paid_in_capital,higher_is_better
6,ft_growth_balance__growth_capital_lease_obliga...,ft_growth_balance,performance,balance_growth,growth_capital_lease_obligations_current,higher_is_better
7,ft_growth_balance__growth_cash_and_cash_equiva...,ft_growth_balance,performance,balance_growth,growth_cash_and_cash_equivalents,higher_is_better
8,ft_growth_balance__growth_cash_and_short_term_...,ft_growth_balance,performance,balance_growth,growth_cash_and_short_term_investments,higher_is_better
9,ft_growth_balance__growth_common_stock,ft_growth_balance,performance,balance_growth,growth_common_stock,higher_is_better


## Predictive-Power Evaluation

The evaluator uses daily cross-sectional rank IC and top-minus-bottom quintile forward-return spread. Direction is normalized before ranking: after scoring, higher feature rank should imply higher expected forward return.


In [7]:
def _rank_2d_nan(values: np.ndarray) -> np.ndarray:
    out = np.full(values.shape, np.nan, dtype="float32")
    for i in range(values.shape[0]):
        row = values[i]
        valid = np.isfinite(row)
        count = int(valid.sum())
        if count == 0:
            continue
        order = np.argsort(row[valid], kind="mergesort")
        ranks = np.empty(count, dtype="float32")
        ranks[order] = np.arange(1, count + 1, dtype="float32")
        out[i, np.flatnonzero(valid)] = ranks
    return out


def _mean_center_nan(values: np.ndarray, axis: int) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(values, axis=axis, keepdims=True)
    return values - mean


def _build_eval_arrays(features: list[str]) -> tuple[np.ndarray, dict[int, np.ndarray], pd.DatetimeIndex, list[str]]:
    dates = pd.DatetimeIndex(sorted(panel["date"].dropna().unique()))
    symbols = sorted(panel["symbol"].dropna().unique())
    feature_values = np.full((len(dates), len(symbols), len(features)), np.nan, dtype="float32")
    for feature_idx, feature in enumerate(features):
        pivot = panel.pivot(index="date", columns="symbol", values=feature).reindex(index=dates, columns=symbols)
        feature_values[:, :, feature_idx] = pivot.to_numpy(dtype="float32")

    direction_sign = feature_metadata.set_index("feature").loc[features, "expected_direction"].map({"higher_is_better": 1.0, "lower_is_better": -1.0}).to_numpy(dtype="float32")
    feature_scores = feature_values * direction_sign.reshape(1, 1, -1)
    returns_by_horizon = {}
    for horizon in HORIZONS:
        pivot = panel.pivot(index="date", columns="symbol", values=f"forward_return_{horizon}d").reindex(index=dates, columns=symbols)
        returns_by_horizon[horizon] = pivot.to_numpy(dtype="float32")
    return feature_scores, returns_by_horizon, dates, symbols


def evaluate_features(features: list[str]) -> pd.DataFrame:
    start = perf_counter()
    feature_scores, returns_by_horizon, _, _ = _build_eval_arrays(features)
    days, symbols, feature_count = feature_scores.shape
    flat_features = feature_scores.transpose(0, 2, 1).reshape(days * feature_count, symbols)
    feature_ranks = _rank_2d_nan(flat_features).reshape(days, feature_count, symbols).transpose(0, 2, 1)
    centered_features = _mean_center_nan(feature_ranks, axis=1)
    rows = []
    for horizon in HORIZONS:
        returns = returns_by_horizon[horizon]
        return_ranks = _rank_2d_nan(returns)
        centered_returns = _mean_center_nan(return_ranks, axis=1)
        valid = np.isfinite(centered_features) & np.isfinite(centered_returns[:, :, None])
        cf = np.where(valid, centered_features, 0.0)
        cr = np.where(np.isfinite(centered_returns), centered_returns, 0.0)
        numerator = np.einsum("dsf,ds->df", cf, cr)
        denom = np.sqrt(np.einsum("dsf,dsf->df", cf, cf) * np.einsum("ds,ds->d", cr, cr)[:, None])
        daily_ic = numerator / denom
        daily_ic[denom == 0] = np.nan

        for feature_idx, feature in enumerate(features):
            feature_score = feature_scores[:, :, feature_idx]
            ret = returns
            valid_pair = np.isfinite(feature_score) & np.isfinite(ret)
            obs = int(valid_pair.sum())
            if obs < MIN_OBS:
                continue
            spreads = []
            for day_idx in range(days):
                day_valid = valid_pair[day_idx]
                if int(day_valid.sum()) < 10:
                    continue
                scores = feature_score[day_idx, day_valid]
                rets = ret[day_idx, day_valid]
                lo = np.nanquantile(scores, 0.2)
                hi = np.nanquantile(scores, 0.8)
                long = rets[scores >= hi]
                short = rets[scores <= lo]
                if len(long) and len(short):
                    spreads.append(float(np.nanmean(long) - np.nanmean(short)))
            ic = daily_ic[:, feature_idx]
            rows.append({
                "feature": feature,
                "horizon": horizon,
                "mean_daily_rank_ic": float(np.nanmean(ic)),
                "median_daily_rank_ic": float(np.nanmedian(ic)),
                "rank_ic_hit_rate": float(np.nanmean(ic > 0)),
                "spread_bps": float(np.nanmedian(spreads) * 10_000) if spreads else np.nan,
                "observations": obs,
            })
    run_timings["evaluation_seconds"] = perf_counter() - start
    return pd.DataFrame(rows)

results_df = evaluate_features(feature_cols)
results_df = results_df.merge(feature_metadata, on="feature", how="left")
print({"evaluation_seconds": round(run_timings["evaluation_seconds"], 2), "feature_horizon_results": len(results_df)})
display(results_df.sort_values(["horizon", "mean_daily_rank_ic"], ascending=[True, False]).groupby("horizon").head(12))


{'evaluation_seconds': 69.49, 'feature_horizon_results': 696}


,feature,horizon,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations,family,controller,source_section,source_column,expected_direction
195,ft_ratios_valuation__enterprise_value,20,0.0643,0.0618,0.6498,159.4258,236383,ft_ratios_valuation,ratios,metrics,enterprise_value,lower_is_better
205,ft_ratios_valuation__market_cap,20,0.0619,0.0576,0.6437,149.5150,236383,ft_ratios_valuation,ratios,metrics,market_cap,lower_is_better
98,ft_growth_cash__growth_purchase_of_investment_...,20,0.0568,0.0661,0.0897,98.1496,29469,ft_growth_cash,performance,cash_growth,growth_purchase_of_investment_securities,higher_is_better
184,ft_ratios_solvency__interest_debt_per_share,20,0.0538,0.0597,0.5934,166.2934,236383,ft_ratios_solvency,ratios,ratios,interest_debt_per_share,lower_is_better
204,ft_ratios_valuation__invested_capital,20,0.0501,0.0473,0.5845,127.0739,236383,ft_ratios_valuation,ratios,metrics,invested_capital,lower_is_better
119,ft_growth_income__growth_gross_profit_margin,20,0.0488,0.0461,0.0906,213.9379,29469,ft_growth_income,performance,income_growth,growth_gross_profit_margin,higher_is_better
87,ft_growth_cash__growth_net_change_in_cash_and_...,20,0.0478,0.0457,0.0765,38.3890,29469,ft_growth_cash,performance,cash_growth,growth_net_change_in_cash_and_equivalents,higher_is_better
203,ft_ratios_valuation__graham_number,20,0.0463,0.0511,0.6127,154.9293,201531,ft_ratios_valuation,ratios,metrics,graham_number,lower_is_better
107,ft_growth_income__growth_consolidated_net_income,20,0.0426,0.0191,0.0756,217.6335,29469,ft_growth_income,performance,income_growth,growth_consolidated_net_income,higher_is_better
111,ft_growth_income__growth_diluted_earnings_per_...,20,0.0426,0.0182,0.0751,243.9103,29469,ft_growth_income,performance,income_growth,growth_diluted_earnings_per_share,higher_is_better


## Family Results


In [8]:
summary_by_family = (
    results_df.groupby(["horizon", "controller", "family"])
    .agg(
        features=("feature", "nunique"),
        median_rank_ic=("mean_daily_rank_ic", "median"),
        mean_rank_ic=("mean_daily_rank_ic", "mean"),
        median_spread_bps=("spread_bps", "median"),
        positive_ic_share=("mean_daily_rank_ic", lambda s: float((s > 0).mean())),
    )
    .reset_index()
    .sort_values(["horizon", "mean_rank_ic"], ascending=[True, False])
)
display(summary_by_family)

best_by_horizon = summary_by_family.sort_values(["horizon", "mean_rank_ic"], ascending=[True, False]).groupby("horizon").head(1).reset_index(drop=True)
display(best_by_horizon)

stable_families = (
    summary_by_family.groupby(["controller", "family"])
    .agg(
        horizons=("horizon", "nunique"),
        avg_rank_ic=("mean_rank_ic", "mean"),
        min_rank_ic=("mean_rank_ic", "min"),
        avg_spread_bps=("median_spread_bps", "mean"),
        positive_horizons=("mean_rank_ic", lambda s: int((s > 0).sum())),
        avg_positive_ic_share=("positive_ic_share", "mean"),
        features=("features", "max"),
    )
    .reset_index()
    .sort_values(["positive_horizons", "avg_rank_ic"], ascending=[False, False])
)
display(stable_families)


,horizon,controller,family,features,median_rank_ic,mean_rank_ic,median_spread_bps,positive_ic_share
5,20,ratios,ft_ratios_liquidity,7,0.0232,0.0229,49.7568,1.0000
4,20,ratios,ft_ratios_efficiency,5,0.0225,0.0131,49.6448,0.6000
7,20,ratios,ft_ratios_solvency,15,0.0097,0.0065,36.0174,0.6000
2,20,performance,ft_growth_income,38,0.0067,0.0063,26.9273,0.7368
13,20,technicals,ft_technicals_volume,3,0.0049,0.0048,9.1781,1.0000
3,20,performance,ft_performance_price,6,0.0054,0.0041,44.7734,0.6667
0,20,performance,ft_growth_balance,56,0.0003,0.0017,3.2647,0.5179
1,20,performance,ft_growth_cash,50,0.0022,0.0014,5.8451,0.6000
6,20,ratios,ft_ratios_profitability,14,-0.0044,0.0002,-31.7114,0.3571
8,20,ratios,ft_ratios_valuation,24,-0.0052,0.0001,-13.5031,0.3333


,horizon,controller,family,features,median_rank_ic,mean_rank_ic,median_spread_bps,positive_ic_share
0,20,ratios,ft_ratios_liquidity,7,0.0232,0.0229,49.7568,1.0000
1,60,ratios,ft_ratios_liquidity,7,0.0311,0.0349,153.8067,1.0000
2,120,ratios,ft_ratios_liquidity,7,0.0370,0.0413,312.1177,1.0000


,controller,family,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons,avg_positive_ic_share,features
5,ratios,ft_ratios_liquidity,3,0.0330,0.0229,171.8938,3,1.0000,7
4,ratios,ft_ratios_efficiency,3,0.0190,0.0131,107.9864,3,0.6000,5
2,performance,ft_growth_income,3,0.0064,0.0030,54.5346,3,0.7018,38
7,ratios,ft_ratios_solvency,3,0.0063,0.0057,61.9023,3,0.5778,15
8,ratios,ft_ratios_valuation,3,0.0043,0.0001,-26.6031,3,0.4028,24
0,performance,ft_growth_balance,3,0.0039,0.0017,6.6067,3,0.5655,56
3,performance,ft_performance_price,3,0.0021,-0.0036,52.2273,2,0.4444,6
13,technicals,ft_technicals_volume,3,0.0011,-0.0027,-0.1234,2,0.5556,3
1,performance,ft_growth_cash,3,0.0011,-0.0011,11.3023,2,0.5467,50
10,technicals,ft_technicals_momentum,3,0.0005,-0.0021,33.6776,1,0.5000,2


## Written Analysis

This cell is generated from the computed tables above.


In [9]:
def _fmt_seconds(value: float) -> str:
    if pd.isna(value):
        return "nan"
    return f"{value:,.2f}s"

family_counts = feature_metadata.groupby(["controller", "family"]).size().rename("feature_count").reset_index().sort_values(["controller", "family"])
best_lines = [
    f"- {row.horizon}d: `{row.family}` ({row.controller}) with mean rank IC {row.mean_rank_ic:.4f}, median spread {row.median_spread_bps:,.1f} bps, {int(row.features)} features"
    for row in best_by_horizon.itertuples(index=False)
]
stable_top = stable_families.head(8)
stable_lines = [
    f"- `{row.family}`: avg rank IC {row.avg_rank_ic:.4f}, min rank IC {row.min_rank_ic:.4f}, positive horizons {int(row.positive_horizons)}/{int(row.horizons)}, features {int(row.features)}"
    for row in stable_top.itertuples(index=False)
]
weak_bottom = stable_families.tail(5)
weak_lines = [
    f"- `{row.family}`: avg rank IC {row.avg_rank_ic:.4f}, positive horizons {int(row.positive_horizons)}/{int(row.horizons)}"
    for row in weak_bottom.itertuples(index=False)
]

analysis_md = "\n".join([
    f"### {ANALYSIS_LABEL} FinanceToolkit Feature-Family Analysis",
    "",
    f"The notebook completed on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, {len(feature_cols)} features, and {len(family_counts)} FinanceToolkit-style feature families from {panel['date'].min().date()} through {panel['date'].max().date()}.",
    f"Panel construction took {_fmt_seconds(run_timings.get('panel_build_seconds', np.nan))}; evaluation took {_fmt_seconds(run_timings.get('evaluation_seconds', np.nan))}; total measured core runtime was {_fmt_seconds(run_timings.get('panel_build_seconds', 0.0) + run_timings.get('evaluation_seconds', 0.0))}.",
    "",
    "#### Feature-Family Taxonomy",
    *[f"- `{row.family}` ({row.controller}): {int(row.feature_count)} features" for row in family_counts.itertuples(index=False)],
    "",
    "#### Best Family By Horizon",
    *best_lines,
    "",
    "#### Most Stable Families",
    *stable_lines,
    "",
    "#### Weakest Families In This Run",
    *weak_lines,
    "",
    "#### Interpretation",
    "FinanceToolkit's module taxonomy is a useful way to organize reusable feature families, but the family-level signal is not uniform. In this large-cap run, liquidity ratios are the clear strongest family, efficiency ratios are the next clean candidate, and income growth, solvency, and valuation look mild but usable. Broad price-risk and technical-volatility families are weak here, so they should not be promoted from this run alone. Treat these results as a family-selection screen: promote stable families into provider-specific feature generation, then rerun with a larger and less market-cap-constrained universe before making production claims.",
])

display(Markdown(analysis_md))


### OpenBB/FMP screened US >= $100B universe FinanceToolkit Feature-Family Analysis

The notebook completed on 117 symbols, 238,717 symbol-days, 244 features, and 14 FinanceToolkit-style feature families from 2018-01-02 through 2026-06-24.
Panel construction took 6.22s; evaluation took 69.49s; total measured core runtime was 75.71s.

#### Feature-Family Taxonomy
- `ft_growth_balance` (performance): 56 features
- `ft_growth_cash` (performance): 50 features
- `ft_growth_income` (performance): 38 features
- `ft_performance_price` (performance): 6 features
- `ft_ratios_efficiency` (ratios): 5 features
- `ft_ratios_liquidity` (ratios): 7 features
- `ft_ratios_profitability` (ratios): 14 features
- `ft_ratios_solvency` (ratios): 19 features
- `ft_ratios_valuation` (ratios): 31 features
- `ft_risk_price` (risk): 8 features
- `ft_technicals_momentum` (technicals): 2 features
- `ft_technicals_overlap` (technicals): 3 features
- `ft_technicals_volatility` (technicals): 2 features
- `ft_technicals_volume` (technicals): 3 features

#### Best Family By Horizon
- 20d: `ft_ratios_liquidity` (ratios) with mean rank IC 0.0229, median spread 49.8 bps, 7 features
- 60d: `ft_ratios_liquidity` (ratios) with mean rank IC 0.0349, median spread 153.8 bps, 7 features
- 120d: `ft_ratios_liquidity` (ratios) with mean rank IC 0.0413, median spread 312.1 bps, 7 features

#### Most Stable Families
- `ft_ratios_liquidity`: avg rank IC 0.0330, min rank IC 0.0229, positive horizons 3/3, features 7
- `ft_ratios_efficiency`: avg rank IC 0.0190, min rank IC 0.0131, positive horizons 3/3, features 5
- `ft_growth_income`: avg rank IC 0.0064, min rank IC 0.0030, positive horizons 3/3, features 38
- `ft_ratios_solvency`: avg rank IC 0.0063, min rank IC 0.0057, positive horizons 3/3, features 15
- `ft_ratios_valuation`: avg rank IC 0.0043, min rank IC 0.0001, positive horizons 3/3, features 24
- `ft_growth_balance`: avg rank IC 0.0039, min rank IC 0.0017, positive horizons 3/3, features 56
- `ft_performance_price`: avg rank IC 0.0021, min rank IC -0.0036, positive horizons 2/3, features 6
- `ft_technicals_volume`: avg rank IC 0.0011, min rank IC -0.0027, positive horizons 2/3, features 3

#### Weakest Families In This Run
- `ft_technicals_momentum`: avg rank IC 0.0005, positive horizons 1/3
- `ft_technicals_overlap`: avg rank IC -0.0001, positive horizons 1/3
- `ft_ratios_profitability`: avg rank IC -0.0115, positive horizons 1/3
- `ft_technicals_volatility`: avg rank IC -0.0439, positive horizons 0/3
- `ft_risk_price`: avg rank IC -0.0735, positive horizons 0/3

#### Interpretation
FinanceToolkit's module taxonomy is a useful way to organize reusable feature families, but the family-level signal is not uniform. In this large-cap run, liquidity ratios are the clear strongest family, efficiency ratios are the next clean candidate, and income growth, solvency, and valuation look mild but usable. Broad price-risk and technical-volatility families are weak here, so they should not be promoted from this run alone. Treat these results as a family-selection screen: promote stable families into provider-specific feature generation, then rerun with a larger and less market-cap-constrained universe before making production claims.